# L12d Lab: A Linear DQN for Adaptive Drug Dosing
In this lab, we learn a chemotherapy dosing policy for the patient of the [dosing lab](../L12b/CHEME-5800-L12b-Lab-AdaptiveDosing-QLearning-Fall-2026.ipynb) with the deep Q-network (DQN) method. In the dosing lab, value iteration and Q-learning worked on a grid of 41 levels per variable.

Here, the tumor burden and the toxicity stay continuous numbers between 0 and 1. A linear Q-network computes the value of each dose from features of the patient state.

> __Learning Objectives:__
>
> By the end of this lab, you should be able to:
>
> * __Follow grid policies on the continuous patient:__ A grid policy chooses each dose from the patient state rounded to the nearest grid point. We'll follow policies from value iteration on the 41-level grid of the dosing lab and on finer grids, without rounding the patient state.
> * __Build a linear Q-network:__ A linear Q-network computes the value of each dose as a weighted sum of radial basis functions of the tumor burden and the toxicity. We'll build one and evaluate it at any patient state.
> * __Train the network with DQN:__ The DQN algorithm stores the agent's transitions in a replay buffer and fits the network to targets from a delayed copy of itself. We'll train the network in a few seconds and compare its doses with the grid policies.

Let's get started!
___

## Setup, Data, and Prerequisites

First, we load the local [`Include.jl`](Include.jl) setup file and its required packages.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates [`Include.jl`](Include.jl) in the notebook's global scope. The setup file activates the course project, defines local paths, and loads the course package, this lab's code in the [src](src) folder, and the packages used here.

Let's set up our code environment:

In [ ]:
# Load packages and paths from this notebook's local setup file -
include(joinpath(@__DIR__, "Include.jl"));

# Choose one theme; after changing it, rerun this cell and the plotting cells -
theme(:default)   # light background
# theme(:dark)   # dark background

In addition to standard Julia libraries, we'll use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl). Check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types, and data used in this material.

The lab's code is in the [src](src) folder, and it is complete. Each task ends with one question for the class, under __Things to think about__. Some questions ask us to change a value in a code cell and rerun it.
___

## Task 1: Build the Continuous Patient Model
In this task, we build the patient model and treat the patient for one cycle.

The patient state is the pair $(B,Z)$, where $B$ is the tumor burden and $Z$ is the accumulated toxicity, both scaled to lie between 0 and 1. Each treatment cycle, we choose a dose $u$ from $\{0, 1/3, 2/3, 1\}$. As in the dosing lab, the next state is given by:

$$
\begin{align*}
B^{\prime} & = B + g\,B\,(1-B) - k\,u\,B, \\
Z^{\prime} & = Z + b\,u - c\,Z,
\end{align*}
$$

where $g$ is the tumor growth rate, $k$ is the fraction of the tumor killed by a full dose, $b$ is the toxicity added by a full dose, and $c$ is the fraction of the toxicity cleared each cycle. Each new value is clipped to lie between 0 and 1.

The rewards are the same as in the dosing lab:
* A cure, $B^{\prime}\leq 0.10$ with the patient alive, earns 60 and ends the treatment.
* A toxic death, $Z^{\prime}\geq 0.90$, costs 60 and ends the treatment.
* Any other cycle costs $B^{\prime}+Z^{\prime}+0.1\,u$, the tumor and the toxicity left after the cycle plus a small charge for the dose.

The one change from the dosing lab is that $B^{\prime}$ and $Z^{\prime}$ are not rounded to a grid level. Let's build the `patient::MyPatientModel` instance with [the `build(...)` function](src/Factory.jl):

In [ ]:
patient = build(MyPatientModel, (
    doses = [0.0, 1/3, 2/3, 1.0], # the dose u of actions 1 to 4
    g = 0.15,       # tumor growth rate per cycle
    k = 0.35,       # fraction of the tumor killed per cycle at full dose
    b = 0.50,       # toxicity added per cycle at full dose
    c = 0.35,       # fraction of the toxicity cleared per cycle
    Bcure = 0.10,   # cured when B ≤ Bcure
    Zlethal = 0.90, # toxic death when Z ≥ Zlethal
    wB = 1.0,       # penalty weight of the tumor burden B′ after a cycle
    wZ = 1.0,       # penalty weight of the toxicity Z′ after a cycle
    wu = 0.10,      # penalty weight of the dose u
    rcure = 60.0,   # reward for a cure
    rdead = -60.0,  # reward for a toxic death
));
γ = 0.95; # discount factor

[The `treat(...)` function](src/Compute.jl) plays the role of the world. Given the state $(B,Z)$ and an action $a$, it returns the next state $(B^{\prime},Z^{\prime})$, the reward $r$, and the done flag, which is `true` if the patient is cured or has died.

Let's treat a patient with a large tumor and no toxicity, $(B,Z)=(0.90,0)$, for one cycle with each of the four doses:

In [ ]:
let
    B = 0.90; # the starting tumor burden: a large tumor
    Z = 0.0;  # the starting toxicity: none

    # one row per action: the dose, then what the world returns after one cycle -
    one_cycle = DataFrame(action = Int64[], dose = Float64[], B′ = Float64[], Z′ = Float64[], r = Float64[], done = Bool[]);
    for a ∈ 1:4
        (B′, Z′, r, done) = treat(patient, B, Z, a);
        push!(one_cycle, (a, round(patient.doses[a], digits = 2), round(B′, digits = 4), round(Z′, digits = 4), round(r, digits = 4), done));
    end
    pretty_table(one_cycle; backend = :text, fit_table_in_display_horizontally = false)
end

__What do we see?__ A larger dose shrinks the tumor more, but it also adds more toxicity. None of the four doses ends the treatment in one cycle. Unlike in the dosing lab, the next states are not rounded. On the dosing lab's 41-level grid, each one would be rounded to the nearest multiple of 0.025.

### Things to think about

__Question:__ Which dose gives the largest reward in this one cycle? Why would giving that dose in every cycle be a poor treatment?
___

## Task 2: Follow Grid Policies on the Continuous Patient
In this task, we follow two policies from value iteration on the continuous patient model.

[The `solvegrid(...)` function](src/Compute.jl) runs value iteration on a grid, as in the dosing lab, and returns the optimal action at each grid point. With 41 levels, it gives the optimal policy of the dosing lab. With 201 levels, the grid is five times finer.

In the dosing lab, the world rounded the patient state to a grid level after every cycle. In the last cycle, the equations gave a tumor burden of 0.112, and rounding made it 0.100, a cure.

Here, the state is not rounded. Does the same policy still cure the patient?

To choose a dose for a continuous state, a grid policy looks up the nearest grid point and gives its action. Let's compute both grid policies and follow them from the dosing lab's starting state:

In [ ]:
grid_41 = solvegrid(patient; nlevels = 41, γ = γ);   # the dosing lab's grid: 41^2 = 1681 states
grid_201 = solvegrid(patient; nlevels = 201, γ = γ); # a finer grid: 201^2 = 40401 states
run_41 = simulate(patient, grid_41; B₀ = 0.90, Z₀ = 0.0, maxcycles = 100, γ = γ);
run_201 = simulate(patient, grid_201; B₀ = 0.90, Z₀ = 0.0, maxcycles = 100, γ = γ);

let
    # initialize --
    runs = [run_201, run_41]; # the treatment of each policy
    labels = ["201-level grid", "41-level grid"];
    colors = [:seagreen, :darkorange]; # one color per policy
    last_cycle = 40; # the 41-level policy runs to the cycle limit, so we draw only its first cycles
    p1 = plot(); # the tumor burden of each policy
    p2 = plot(); # the toxicity of each policy

    # the tumor burden and the toxicity of each policy, cycle by cycle -
    for j ∈ 1:length(runs)
        n = min(runs[j].cycles, last_cycle); # the last cycle we draw for this policy
        cycles = range(0, stop = n, step = 1) |> collect; # the cycle numbers 0, 1, 2, ...
        plot!(p1, cycles, runs[j].B[1:(n + 1)], lw = 3, c = colors[j], label = labels[j]);
        plot!(p2, cycles, runs[j].Z[1:(n + 1)], lw = 3, c = colors[j], label = labels[j]);
    end

    # the level that ends the treatment in each figure -
    hline!(p1, [patient.Bcure], lw = 2, ls = :dash, c = :gray50, label = "cure level");
    hline!(p2, [patient.Zlethal], lw = 2, ls = :dash, c = :gray50, label = "lethal level");

    # labels and style: the gray panel is see-through, so the figure works in both themes -
    ylabel!(p1, "Tumor burden B");
    ylabel!(p2, "Toxicity Z");
    for p ∈ [p1, p2]
        xlabel!(p, "Treatment cycle");
        plot!(p, background_color_inside = RGBA(0.5, 0.5, 0.5, 0.12), framestyle = :box, fg_legend = :transparent,
            legend = :outertop, legend_column = 2, xlims = (-0.3, last_cycle + 0.3), ylims = (-0.03, 1.05));
    end
    plot(p1, p2, layout = (1, 2), size = (1000, 460), bottom_margin = 5Plots.mm, left_margin = 5Plots.mm)
end

__What do we see?__ The 201-level policy cures the patient. The 41-level policy brings the tumor down to just above the cure level, but it does not cure the patient.

The 41-level policy chooses each dose from the rounded state. A tumor burden just above the cure level, such as 0.109, rounds to the grid level 0.100, which counts as a cure. A cured patient gets no dose, and the tumor grows back.

Let's check the dose that each grid policy gives at the state $(B,Z)=(0.109,0.816)$, which the 41-level policy reaches:

In [ ]:
let
    B = 0.109; # the tumor burden of a state the 41-level policy reaches
    Z = 0.816; # the toxicity of that state

    # the dose each grid policy gives at (B, Z) -
    doses_at_state = DataFrame(policy = ["41-level grid", "201-level grid"],
        dose = round.([patient.doses[gridaction(grid_41, B, Z)], patient.doses[gridaction(grid_201, B, Z)]], digits = 2));
    pretty_table(doses_at_state; backend = :text, fit_table_in_display_horizontally = false)
end

__What do we see?__ At this state, the 41-level policy gives no dose. The 201-level policy gives a dose, and the treatment goes on.

### Things to think about
Let's compute a grid policy for each number of levels in `nlevels_values` and follow each one from the dosing lab's starting state:

In [ ]:
let
    nlevels_values = [41, 49, 81, 201]; # the number of grid levels on each axis

    # for each grid: value iteration, then follow its policy on the patient without a grid -
    grids = DataFrame(levels = Int64[], outcome = Symbol[], cycles = Int64[]);
    for n ∈ nlevels_values
        grid_n = solvegrid(patient; nlevels = n, γ = γ);
        run_n = simulate(patient, grid_n; B₀ = 0.90, Z₀ = 0.0, maxcycles = 100, γ = γ);
        push!(grids, (n, run_n.outcome, run_n.cycles));
    end
    pretty_table(grids; backend = :text, fit_table_in_display_horizontally = false)
end

__Question:__ Which grids cure the patient? Add a few values to `nlevels_values` and rerun the cell: does a finer grid always give a better treatment?
___

## Task 3: Train a Linear DQN
In this task, we build a linear Q-network, train it with DQN, and compare its greedy policy with the grid policies.

The Q-value of action $a$ in the state $s=(B,Z)$ is the discounted return the agent expects after taking action $a$. The network computes it as a weighted sum of features, which is given by:

$$
\begin{equation*}
Q_{\mathbf{W}}(s,a) = \mathbf{w}_{a}^{\top}\phi(s),
\end{equation*}
$$

where $\mathbf{w}_{a}^{\top}$ is row $a$ of the weight matrix $\mathbf{W}$, and $\phi(s)$ is the feature vector of the state. Our features are radial basis functions. We place the centers $\mathbf{c}_{j}$ of $m\times m$ basis functions on an even grid over the square $0\leq B\leq 1$, $0\leq Z\leq 1$. Basis function $j$ is given by:

$$
\begin{equation*}
\phi_{j}(s)=\exp\left(-\frac{\lVert s-\mathbf{c}_{j}\rVert_{2}^{2}}{2\sigma^{2}}\right),
\end{equation*}
$$

where $\lVert s-\mathbf{c}_{j}\rVert_{2}$ is the Euclidean distance from the state to center $j$, and $\sigma$ is the width of every basis function. A last feature is the constant 1, which lets each action's Q-value shift up or down by the same amount in every state. Each action then has $m^{2}+1$ weights.

The grid of centers only places the basis functions. The network can evaluate any state, not only the centers. Let's build the network with [the `build(...)` function](src/Factory.jl), with all its weights at zero.

Let's also plot the basis functions along the tumor burden axis at zero toxicity, for our width and for a narrow and a wide width. The dashed line marks the state $(B,Z)=(0.90,0)$, and the dots mark the value of each basis function there:

In [ ]:
network = build(MyLinearQNetworkModel, (nactions = 4, m = 6, σ = 0.15)); # 4 actions, 6 × 6 basis functions of width 0.15
φ = features(network, 0.90, 0.0); # the features of the state B = 0.90, Z = 0

# the basis functions along the tumor burden axis at Z = 0, for a narrow width, our width, and a wide width -
let
    B_values = range(0.0, 1.0, length = 201) |> collect; # tumor burdens from 0 to 1
    row = findall(network.centers[:, 2] .== 0.0); # the basis functions with centers on the row Z = 0
    panels = [];
    for σ ∈ [0.05, network.σ, 0.5]
        network_σ = build(MyLinearQNetworkModel, (nactions = 4, m = 6, σ = σ));
        φ_values = [features(network_σ, B, 0.0) for B ∈ B_values]; # the features at each tumor burden
        p = plot(xlabel = "tumor burden B (Z = 0)", ylabel = "basis function value", ylims = (0, 1.05),
            title = "σ = $(σ)", titlefontsize = 11, legend = false);
        for j ∈ row
            plot!(p, B_values, [φ_B[j] for φ_B ∈ φ_values], lw = 2, c = :dodgerblue3);
        end
        vline!(p, [0.90], c = :gray, ls = :dash); # the state B = 0.90
        scatter!(p, fill(0.90, length(row)), features(network_σ, 0.90, 0.0)[row], c = :red, ms = 4);
        push!(panels, p);
    end
    plot(panels..., layout = (1, 3), size = (960, 320), margin = 5Plots.mm)
end

__What do we see?__ Each basis function is a bump that equals 1 at its center. With our width, only the bumps centered near the state are large at the state, and the rest are close to zero. A gradient step at this state then changes mostly the weights of the nearby bumps.

The width sets how far each bump reaches. A narrow width leaves gaps between the bumps, and a wide width makes every bump reach across most of the axis.

### Train the network
[The `mydqn(...)` function](src/Compute.jl) carries out the DQN algorithm of the lecture. Each episode is one simulated treatment that starts the patient at a random state. Random starts spread the agent's experience over many states.

The keyword arguments of `mydqn(...)` set the training:
* `episodes` is the number of simulated treatments, and `maxcycles` is the most cycles in each one.
* `ϵ_start` and `ϵ_floor` set the exploration probability $\epsilon$, the chance that a cycle gives a random dose. It falls in a straight line from `ϵ_start` and then stays at `ϵ_floor`.
* `capacity` is the size of the replay buffer. Training starts once the buffer holds `warmup` transitions.
* `batchsize` is the mini-batch size $N_{\text{batch}}$, and `α` is the learning rate of each gradient step.
* `C` is the number of cycles between copies of the Q-network to the target network.

Transition $i$ of a mini-batch holds the state $s_{i}$, the action $a_{i}$, the reward $r_{i}$, the next state $s_{i}^{\prime}$, and the done flag $d_{i}$, which is 1 for a cure or a toxic death and 0 otherwise. Its target $y_{i}$ is the value that the prediction $\mathbf{w}_{a_{i}}^{\top}\phi(s_{i})$ should match:

$$
\begin{equation*}
y_{i}=r_{i}+\gamma\,(1-d_{i})\max_{a^{\prime}}Q_{\mathbf{W}^{-}}(s_{i}^{\prime},a^{\prime}),
\end{equation*}
$$

where the maximum runs over the four actions $a^{\prime}$ in the next state. The factor $(1-d_{i})$ drops the future value when the treatment has ended. $\mathbf{W}^{-}$ holds the weights of the target network, which keeps the targets fixed between copies.

One gradient step changes the weights to shrink the prediction errors $y_{i}-\mathbf{w}_{a_{i}}^{\top}\phi(s_{i})$. For the linear network, the step updates the row of each action in the mini-batch. The update of row $a$ is given by:

$$
\begin{equation*}
\mathbf{w}_{a}\gets\mathbf{w}_{a}+\frac{\alpha}{N_{\text{batch}}}\sum_{i\,:\,a_{i}=a}\left(y_{i}-\mathbf{w}_{a}^{\top}\phi(s_{i})\right)\phi(s_{i}),
\end{equation*}
$$

where the sum runs over the transitions $i$ of the mini-batch that took action $a$. This step is gradient descent on the loss of the lecture:

$$
\begin{equation*}
\mathcal{L}(\mathbf{W})=\frac{1}{2N_{\text{batch}}}\sum_{i=1}^{N_{\text{batch}}}\left(y_{i}-\mathbf{w}_{a_{i}}^{\top}\phi(s_{i})\right)^{2}.
\end{equation*}
$$

The `mydqn(...)` function records the loss of each gradient step, computed before the step. Let's train the network and plot the loss against the gradient step, on a log scale:

In [ ]:
result = mydqn(network, patient; episodes = 3000, maxcycles = 80, γ = γ, α = 0.05,
    capacity = 20_000, batchsize = 32, warmup = 1000, C = 500, ϵ_start = 1.0, ϵ_floor = 0.05, seed = 1234);
trained = result.network; # the trained network

# the loss of each gradient step, and its average over blocks of gradient steps -
let
    loss = result.loss; # the mini-batch loss of each gradient step
    steps = range(1, stop = length(loss), step = 1) |> collect; # the gradient steps 1, 2, 3, ...
    block = 500; # the number of gradient steps in one block
    number_of_blocks = length(loss) ÷ block;
    block_steps = [k*block for k ∈ 1:number_of_blocks]; # the last gradient step of each block
    block_loss = [mean(loss[((k - 1)*block + 1):(k*block)]) for k ∈ 1:number_of_blocks]; # the average loss of each block

    # the loss of each step in gray, and the block average on top -
    plot(steps, loss, lw = 1, c = :gray50, alpha = 0.35, label = "loss of each gradient step");
    plot!(block_steps, block_loss, lw = 3, c = :dodgerblue3, label = "average over $(block) steps");

    # labels and style: the gray panel is see-through, so the figure works in both themes -
    xlabel!("Gradient step");
    ylabel!("Mini-batch loss");
    plot!(yscale = :log10, ylims = (0.2, 400), yticks = ([1, 10, 100], ["1", "10", "100"]), xformatter = :plain,
        background_color_inside = RGBA(0.5, 0.5, 0.5, 0.12), framestyle = :box, fg_legend = :transparent,
        legend = :outertop, legend_column = 2, size = (760, 460));
end

__What do we see?__ The loss of a single step jumps up and down, because each mini-batch holds different random transitions. The average loss falls over the training run. It is still falling at the end of the run.

Would a longer run lower the loss further? Let's train with more episodes and plot the average loss of each run:

In [ ]:
let
    episodes_values = [3000, 6000, 12000]; # the number of episodes of each training run
    colors = [:dodgerblue3, :seagreen, :darkorange]; # one color per training run
    block = 500; # the number of gradient steps in one block

    # for each run: train a network, then plot the average loss of each block of gradient steps -
    plot();
    for j ∈ 1:length(episodes_values)
        result_j = mydqn(network, patient; episodes = episodes_values[j], maxcycles = 80, γ = γ, α = 0.05,
            capacity = 20_000, batchsize = 32, warmup = 1000, C = 500, ϵ_start = 1.0, ϵ_floor = 0.05, seed = 1234);
        loss = result_j.loss; # the mini-batch loss of each gradient step
        number_of_blocks = length(loss) ÷ block;
        block_steps = [k*block for k ∈ 1:number_of_blocks]; # the last gradient step of each block
        block_loss = [mean(loss[((k - 1)*block + 1):(k*block)]) for k ∈ 1:number_of_blocks]; # the average loss of each block
        plot!(block_steps, block_loss, lw = 3, c = colors[j], label = "$(episodes_values[j]) episodes");
    end

    # labels and style: the gray panel is see-through, so the figure works in both themes -
    xlabel!("Gradient step");
    ylabel!("Average mini-batch loss");
    plot!(yscale = :log10, ylims = (1.5, 100), yticks = ([2, 5, 10, 20, 50], ["2", "5", "10", "20", "50"]), xformatter = :plain,
        background_color_inside = RGBA(0.5, 0.5, 0.5, 0.12), framestyle = :box, fg_legend = :transparent,
        legend = :outertop, legend_column = 3, size = (760, 460));
end

__What do we see?__ A longer run lowers the loss a little more, and then the loss levels off. It does not fall to zero. The targets change at each copy to the target network, and a weighted sum of basis functions cannot match every target exactly.

A longer run also explores for more episodes, because the exploration rate falls over a fixed fraction of the episodes. Its loss falls more slowly.

### Follow the learned policy
The learned policy gives each state the dose with the largest Q-value, with no exploration. [The `simulate(...)` function](src/Compute.jl) follows it from a starting state until the patient is cured, dies of toxicity, or a cycle limit has passed.

Let's follow the learned policy from the dosing lab's starting state, $(B,Z)=(0.90,0)$, and draw the tumor burden, the toxicity, and the dose over the treatment cycles:

In [ ]:
run_dqn = simulate(patient, trained; B₀ = 0.90, Z₀ = 0.0, maxcycles = 100, γ = γ);

let
    cycles = range(0, stop = run_dqn.cycles, step = 1) |> collect; # the cycle numbers 0, 1, 2, ...
    doses = [run_dqn.dose; run_dqn.dose[end]]; # the dose of each cycle; the last value closes the step line

    # the tumor burden and the toxicity after each cycle -
    plot(cycles, run_dqn.B, lw = 3, c = :dodgerblue3, marker = :circle, ms = 4, msc = :dodgerblue3, label = "tumor burden B");
    plot!(cycles, run_dqn.Z, lw = 3, c = :red, marker = :circle, ms = 4, msc = :red, label = "toxicity Z");

    # the dose in each cycle, in the color of the legend text: black on the light theme, white on the dark theme -
    dose_color = current()[1][:legend_font_color];
    plot!(cycles, doses, seriestype = :steppost, lw = 2, ls = :dash, c = dose_color, label = "dose u");

    # the levels that end the treatment -
    hline!([patient.Bcure], lw = 2, ls = :dash, c = :dodgerblue3, label = "cure level");
    hline!([patient.Zlethal], lw = 2, ls = :dash, c = :red, label = "lethal level");

    # labels and style: the gray panel is see-through, so the figure works in both themes -
    xlabel!("Treatment cycle");
    ylabel!("Tumor burden, toxicity, and dose");
    plot!(background_color_inside = RGBA(0.5, 0.5, 0.5, 0.12), framestyle = :box, fg_legend = :transparent,
        legend = :outertop, legend_column = 3, xlims = (-0.3, run_dqn.cycles + 0.3), ylims = (-0.03, 1.05), size = (760, 460));
end

__What do we see?__ The learned policy cures the patient without a toxic death. It gives mostly moderate doses and only a few full doses, never two in a row. The toxicity rises after each large dose and falls after each small one, and it stays below the lethal level while the tumor shrinks.

### Compare with the grid policies
Let's plot the treatment of the linear DQN with the treatments of the two grid policies, from the same starting state:

In [ ]:
let
    # initialize --
    runs = [run_dqn, run_201, run_41]; # the treatment of each policy
    labels = ["linear DQN", "201-level grid", "41-level grid"];
    colors = [:dodgerblue3, :seagreen, :darkorange]; # one color per policy
    last_cycle = 40; # the 41-level policy runs to the cycle limit, so we draw only its first cycles
    p1 = plot(); # the tumor burden of each policy
    p2 = plot(); # the toxicity of each policy

    # the tumor burden and the toxicity of each policy, cycle by cycle -
    for j ∈ 1:length(runs)
        n = min(runs[j].cycles, last_cycle); # the last cycle we draw for this policy
        cycles = range(0, stop = n, step = 1) |> collect; # the cycle numbers 0, 1, 2, ...
        plot!(p1, cycles, runs[j].B[1:(n + 1)], lw = 3, c = colors[j], label = labels[j]);
        plot!(p2, cycles, runs[j].Z[1:(n + 1)], lw = 3, c = colors[j], label = labels[j]);
    end

    # the level that ends the treatment in each figure -
    hline!(p1, [patient.Bcure], lw = 2, ls = :dash, c = :gray50, label = "cure level");
    hline!(p2, [patient.Zlethal], lw = 2, ls = :dash, c = :gray50, label = "lethal level");

    # labels and style: the gray panel is see-through, so the figure works in both themes -
    ylabel!(p1, "Tumor burden B");
    ylabel!(p2, "Toxicity Z");
    for p ∈ [p1, p2]
        xlabel!(p, "Treatment cycle");
        plot!(p, background_color_inside = RGBA(0.5, 0.5, 0.5, 0.12), framestyle = :box, fg_legend = :transparent,
            legend = :outertop, legend_column = 2, xlims = (-0.3, last_cycle + 0.3), ylims = (-0.03, 1.05));
    end
    plot(p1, p2, layout = (1, 2), size = (1000, 460), bottom_margin = 5Plots.mm, left_margin = 5Plots.mm)
end

__What do we see?__ The linear DQN and the 201-level policy both cure the patient. The 201-level policy needs fewer cycles. Its toxicity also comes closer to the lethal level.

### Things to think about
The width $\sigma$ sets how far each basis function reaches from its center. Let's train one network for each width in `σ_values` and follow each learned policy from the same start:

In [ ]:
let
    σ_values = [0.05, 0.15, 0.5]; # the widths to compare: narrow, the width we trained with, and wide

    # for each width: build and train a network, then follow its greedy policy -
    widths = DataFrame(σ = Float64[], outcome = Symbol[], cycles = Int64[]);
    for σ ∈ σ_values
        network_σ = build(MyLinearQNetworkModel, (nactions = 4, m = 6, σ = σ));
        result_σ = mydqn(network_σ, patient; episodes = 3000, maxcycles = 80, γ = γ, α = 0.05,
            capacity = 20_000, batchsize = 32, warmup = 1000, C = 500, ϵ_start = 1.0, ϵ_floor = 0.05, seed = 1234);
        run_σ = simulate(patient, result_σ.network; B₀ = 0.90, Z₀ = 0.0, maxcycles = 100, γ = γ);
        push!(widths, (σ, run_σ.outcome, run_σ.cycles));
    end
    pretty_table(widths; backend = :text, fit_table_in_display_horizontally = false)
end

__Question:__ Which widths cure the patient? Use the basis function figure to explain why a width that is too narrow or too wide fails.
___

## Summary
In this lab, we learned a dosing policy for the patient of the dosing lab with a linear DQN, without rounding the patient state to a grid.

> __Key Takeaways:__
>
> * __Rounding can give the wrong dose:__ We followed grid policies from value iteration on the patient without a grid. The dosing lab's grid rounded a tumor just above the cure level down to a cure and stopped the treatment, and one finer grid gave a lethal dose.
>
> * __A linear Q-network replaces the table:__ We built a Q-network that computes the value of each dose as a weighted sum of radial basis functions of the tumor burden and the toxicity. It gives a dose for every patient state, with no grid and no rounding, and it needs far fewer numbers than a Q-table on a fine grid.
>
> * __Training the network with DQN:__ We trained the network on simulated treatments, with a replay buffer and a target network. The greedy policy cured the patient from the dosing lab's starting state and kept the toxicity below the lethal level.

DQN helps us with continuous states, but the actions are still discrete. This is often fine, because many decisions are categorical. For continuous states and actions, we need other methods, such as the policy gradient and actor-critic methods of [Sutton and Barto, 2018, Chapter 13](http://incompleteideas.net/book/the-book-2nd.html).

___